In [9]:
import os
import shutil
import random
import torch
import torchaudio
import torchaudio.transforms as tt
import torch.nn.functional as ff

In [13]:
wham_root = r"/home/ovistetom/Documents/Databases_Local/WHAM"

In [14]:
SR = 16000

In [15]:
wham_path = os.path.join(wham_root, 'wham_noise', 'cv')
wham_list = os.listdir(wham_path)
fade_len_in_s = 0.25
fade_len = int(SR * fade_len_in_s)
fade_transform = tt.Fade(fade_in_len=fade_len, fade_out_len=fade_len, fade_shape='linear')
for i, (file_name_1, file_name_2, file_name_3, file_name_4) in enumerate(zip(wham_list[::4], wham_list[1::4], wham_list[2::4], wham_list[3::4])):
    # Load audio files.
    file_path_1 = os.path.join(wham_path, file_name_1)
    file_path_2 = os.path.join(wham_path, file_name_2)
    file_path_3 = os.path.join(wham_path, file_name_3)
    file_path_4 = os.path.join(wham_path, file_name_4)
    audio_1, sr_1 = torchaudio.load(file_path_1, channels_first=True)
    audio_2, sr_2 = torchaudio.load(file_path_2, channels_first=True)
    audio_3, sr_3 = torchaudio.load(file_path_3, channels_first=True)
    audio_4, sr_4 = torchaudio.load(file_path_4, channels_first=True)
    assert sr_1 == sr_2 == sr_3 == sr_4 == SR, "Audio files must have the same sample rate."
    # Keep only first channel and first 4s of each audio file (pad if necessary).
    audio_1 = audio_1[0, :4*SR] if audio_1.size(1) > 4*SR else ff.pad(audio_1, (0, 4*SR-audio_1.size(1)), mode='reflect')[0]
    audio_2 = audio_2[0, :4*SR] if audio_2.size(1) > 4*SR else ff.pad(audio_2, (0, 4*SR-audio_2.size(1)), mode='reflect')[0]
    audio_3 = audio_3[0, :4*SR] if audio_3.size(1) > 4*SR else ff.pad(audio_3, (0, 4*SR-audio_3.size(1)), mode='reflect')[0]
    audio_4 = audio_4[0, :4*SR] if audio_4.size(1) > 4*SR else ff.pad(audio_4, (0, 4*SR-audio_4.size(1)), mode='reflect')[0]
    # Normalize audio files.
    audio_1 /= torch.max(torch.abs(audio_1))
    audio_2 /= torch.max(torch.abs(audio_2))
    audio_3 /= torch.max(torch.abs(audio_3))
    audio_4 /= torch.max(torch.abs(audio_4))
    # Create 4-channel segment of length 4s.
    segment_4s = torch.stack((audio_1, audio_2, audio_3, audio_4), dim=0)
    segment_4s = fade_transform(segment_4s)
    # Save segment.
    new_path = os.path.join(wham_root, 'sliced_wham', 'val')
    os.makedirs(new_path, exist_ok=True)
    file_path_dst = os.path.join(new_path, f"{i:05}.wav")
    torchaudio.save(file_path_dst, segment_4s, sr_1)